# Автоматическая классификация фразеологизмов о смерти в многоязычном корпусе.

## Задача
Цель работы – проверить, насколько реально автоматизировать классификацию фразеологизмов с погребальной семантикой по типам метафорических переносов.

## Зачем?
Ручная разметка корпуса занимает много времени и ужасно масштабируется. При расширении корпуса (например, добавлении других языков или даже просто при добавлении словарей) возникает жизненно важная потребность хоть как-то автоматизировать этот процесс.

## Подход
В работе сравниваются:
- TF-IDF + Logistic Regression (baseline)
- BERT (XLM-RoBERTa)
- Бонус: перенос модели, обученной на русском языке, на английский и сербский.

## Данные

Корпус включает фразеологизмы, размеченные по типам репрезентации:
- пространственно-переходная модель;
- акватическая метафорика;
- религиозная репрезентация;
- хтоническая репрезентация;
- семантика насильственного взаимодействия
и др.

Разметка является многометочной.

## Предобработка
- унификация формата данных из разных языков;
- нормализация меток;
- объединение текста и дефиниции;
- удаление шумовых и неполных данных.

In [1]:
import pandas as pd
import numpy as np

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from sklearn.multiclass import OneVsRestClassifier

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [2]:
# пути
path_sr = r"C:\Users\User\Питон Вышка все файлы\словари\txt\српске\смрт\done\Српске_руч.csv"
path_en = r"C:\Users\User\Питон Вышка все файлы\словари\txt\англ\смрт\done\english_death_handmade.csv"
path_ru = r"C:\Users\User\Питон Вышка все файлы\словари\txt\РЯ\смрт\done\Русская_смерть_ручная.csv"

# загрузка
df_sr = pd.read_csv(path_sr)
df_en = pd.read_csv(path_en)
df_ru = pd.read_csv(path_ru)

def prepare_df(df, lang):
    return pd.DataFrame({
        "text": df["Фразеологизм"],
        "definition": df["Дефиниция"],
        "labels": df["Категория"],
        "lang": lang
    })

# приведение к формату
df_sr = prepare_df(df_sr, "sr")
df_en = prepare_df(df_en, "en")
df_ru = prepare_df(df_ru, "ru")

# объединение
df = pd.concat([df_ru, df_en, df_sr], ignore_index=True)

print("Общий размер:", len(df))
df.head()

Общий размер: 986


,text,definition,labels,lang
0,"Умереть на печи — все равно, что с перепою.",NaN,Народно-поэтическая репрезентация; Социально-э...,ru
1,Грешно дать умереть младенцу в люльке: все одн...,NaN,Религиозная репрезентация; Семантика насильств...,ru
2,У боярина семь дочерей: будет из них и смерть ...,NaN,Народно-поэтическая репрезентация; Фаталистиче...,ru
3,"Солдату умереть в поле, матросу в море.",NaN,Семантика насильственного взаимодействия; Хтон...,ru
4,"Жалеть не помочь, когда смерть пришла.",NaN,Пространственно-переходная модель; Темпоральна...,ru


In [3]:
#переименовываю покороче
label_map = {
    "Семантика насильственного взаимодействия": "насилие",
    "Темпоральная репрезентация": "время",
    "Религиозная репрезентация": "религия",
    "Соматическая репрезентация": "тело",
    "Пространственно-переходная модель": "переход",
    "Аксиологически позитивная репрезентация": "позитивное",
    "Фаталистическая репрезентация": "неизбежность",
    "Персонификация смерти": "персона",
    "Криомортальная семантика": "мороз",
    "Акватическая метафорика": "вода",
    "Народно-поэтическая репрезентация": "фольклор",
    "Хтоническая репрезентация": "земля",
    "Античное наследие (Греция и Рим)": "античка",
    "Дидактика": "воспитание",
    "Телесная деструкция": "разрушение тела"
}

EXCLUDE = {"Социально-этическая семантика"} # слишком абстрактная и требует доработки.


def clean_labels(x):
    if pd.isna(x):
        return None

    parts = (
        str(x)
        .replace("\t", ";")
        .replace(",", ";")
        .split(";")
    )

    parts = [p.strip() for p in parts if p.strip()]
    parts = [label_map.get(p, p) for p in parts]
    parts = [p for p in parts if p not in EXCLUDE]

    return list(dict.fromkeys(parts)) if parts else None


df["labels_list"] = df["labels"].apply(clean_labels)

In [4]:
# убрала все, где нет самой ФЕ или меток, а также кривой текст
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


df["text"] = df["text"].apply(clean_text)

df = df.dropna(subset=["text", "labels_list"])
df = df.drop_duplicates(subset=["text"])

print("После очистки:", len(df))

После очистки: 944


In [5]:
# фильтрация (только русский корпус)
df_ru = df[df["lang"] == "ru"].copy()

print("Размер RU корпуса:", len(df_ru))

# проверка распределения меток
all_labels = [label for labels in df_ru["labels_list"] for label in labels]

print("Уникальных меток:", len(set(all_labels)))
print("Топ-10 меток:", Counter(all_labels).most_common(10))

Размер RU корпуса: 548
Уникальных меток: 15
Топ-10 меток: [('неизбежность', 176), ('фольклор', 134), ('переход', 116), ('тело', 110), ('земля', 97), ('насилие', 79), ('религия', 53), ('позитивное', 50), ('время', 49), ('персона', 38)]


# TF-IDF модель

Baseline: TF-IDF + Logistic Regression.

TF-IDF используется как простой, интерпретируемый и устойчивый метод для задач с ограниченным объемом данных. В теории, он должен хорошо работать для категорий, имеющих явные лексические маркеры ("Бог" - религия, "земля" - хтонь и т.д.).

Пока попробуем обучить без дефиниций.

In [6]:
# разделение 80/20
train_df, test_df = train_test_split(
    df_ru,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 438
Test: 110


In [7]:
# перевод в бинарный формат тк задача многометочной классификации
mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["labels_list"])
y_test = mlb.transform(test_df["labels_list"])

print("Количество классов:", len(mlb.classes_))

Количество классов: 15


In [8]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

print("Размер матрицы признаков:", X_train.shape)

Размер матрицы признаков: (438, 307)


In [9]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        C=2,
        class_weight="balanced",
        n_jobs=-1
    )
)

model.fit(X_train, y_train)

,estimator,"LogisticRegre...00, n_jobs=-1)"
,n_jobs,None
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,2
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None


In [10]:
y_pred = model.predict(X_test)

f1_micro = f1_score(y_test, y_pred, average="micro")

print("Baseline F1 micro:", round(f1_micro, 3))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

Baseline F1 micro: 0.473

Classification report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.00      0.00      0.00         1
           2       0.07      0.25      0.11         4
           3       0.47      0.70      0.56        10
           4       0.61      0.70      0.65        20
           5       0.00      0.00      0.00         2
           6       0.23      0.30      0.26        10
           7       0.54      0.67      0.60        39
           8       0.94      0.53      0.68        30
           9       0.38      0.80      0.52        10
          10       0.57      0.57      0.57        14
          11       0.25      0.50      0.33         4
          12       0.17      0.18      0.17        11
          13       0.40      0.42      0.41        19
          14       0.42      0.46      0.44        28

   micro avg       0.43      0.53      0.47       204
   macro avg       0.34      0.

### Первые результаты:
- модель угадывает примерно 47% меток правильно
- есть классы с нулями -- вообще не предсказывает, так как мало примеров для обучения
- у частых классов неплохой результат (7 и 8 60/68 соотв.)
- хорошо работает на частых классах, плохо на редких 

## Эксперимент: влияние дефиниций

Некоторые дефиниции расписаны достаточно подробно (напр.: "быть близким к смерти, доживать последние дни своей жизни (о старых или безнадежно больных людях"), но могут и мешать, потому как чаще всего это: "умереть", "погибнуть", "скончаться".

Интересно посмотреть, как изменятся результаты, если добавить в обучение и их.



In [12]:
df_def = df.dropna(subset=["text", "definition", "labels", "lang"]).copy()

df_def["text"] = df_def["text"].astype(str)
df_def["definition"] = df_def["definition"].astype(str)
df_def["labels"] = df_def["labels"].astype(str)
df_def["lang"] = df_def["lang"].astype(str)

# объединение
df_def["input_text"] = (df_def["text"] + " " + df_def["definition"]).str.strip()

# метки
df_def["labels_list"] = df_def["labels"].apply(
    lambda x: [label.strip() for label in x.split(";") if label.strip()]
)

df_def = df_def[df_def["input_text"].str.len() > 0]
df_def = df_def[df_def["labels_list"].map(len) > 0]

# только русский
df_def_ru = df_def[df_def["lang"] == "ru"].copy()

# split
train_df_def, test_df_def = train_test_split(
    df_def_ru,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# binarizer
mlb_def = MultiLabelBinarizer()
y_train_def = mlb_def.fit_transform(train_df_def["labels_list"])
y_test_def = mlb_def.transform(test_df_def["labels_list"])

# TF-IDF
vectorizer_def = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

X_train_def = vectorizer_def.fit_transform(train_df_def["input_text"])
X_test_def = vectorizer_def.transform(test_df_def["input_text"])

# модель
model_def = OneVsRestClassifier(
    LogisticRegression(
        max_iter=3000,
        C=3,
        class_weight="balanced",
        n_jobs=-1
    )
)

model_def.fit(X_train_def, y_train_def)

y_pred_def = model_def.predict(X_test_def)

f1_micro_def = f1_score(y_test_def, y_pred_def, average="micro")

print("TF-IDF + definitions F1 micro:", round(f1_micro_def, 3))

TF-IDF + definitions F1 micro: 0.577


### Результаты сравнения

| Модель | F1 micro |
|--------|---------|
| TF-IDF (только ФЕ) | **0.47** |
| TF-IDF (ФЕ + дефиниции) | **0.58** |

### Вывод

Добавление дефиниций значительно повышает качество модели.

Это можно объяснить тем, что:
- часть из них все-таки содержит явные лексические маркеры категорий;
- модель начинает опираться на "подсказки", а не на сами выражения.

### Выводы по TF-IDF

- F1 micro: **0.47**
- F1 macro: **0.35**

- Модель хорошо работает для частотных категорий
- Плохо предсказывает редкие классы (многие F1 ≈ 0)
- Macro значительно ниже micro -> выраженный дисбаланс классов
- Модель использует частоты слов (мешок слов), без учета контекста

### Ключевое наблюдение

Добавление дефиниций увеличивает качество до **F1 micro ≈ 0.58**.

Это показывает, что:
- модель сильно зависит от лексических подсказок;
- без дефиниций задача становится значительно сложнее.

### Вывод

TF-IDF является хорошим baseline, но:
- не улавливает семантику выражений;
- чувствителен к дополнительной информации в тексте.

Это оправдывает использование моделей, учитывающих контекст (BERT)...

# Модель на основе трансформера (XLM-RoBERTa)

Для более сложного базового уровня используется мультиязычная модель XLM-RoBERTa, способная учитывать контекст и семантику текста.

В отличие от TF-IDF, трансформер:
- учитывает порядок слов и контекст
- способен обрабатывать редкие и сложные формулировки
- потенциально лучше работает с редкими классами

## В данном блоке:
- тексты токенизируются с помощью XLM-RoBERTa tokenizer
- создаётся PyTorch Dataset
- модель дообучается для многометочной классификации
- применяется сигмоида для получения вероятностей
- считаются метрики F1 micro и F1 macro

Главный вопрос: "А если модель понимает контекст, будет ли она лучше, чем глупенький TF-IDF?"

In [19]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=128
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

In [20]:
# использую исходный текст
train_df["input_text"] = train_df["text"] #может ли трансформер сам понять смысл выражения без подсказок?
test_df["input_text"] = test_df["text"]

train_dataset = TextDataset(train_df["input_text"], y_train)
test_dataset = TextDataset(test_df["input_text"], y_test)

In [21]:
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="no",
    logging_steps=50
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.310659
100,0.299611
150,0.297105
200,0.297475
250,0.288599


TrainOutput(global_step=275, training_loss=0.2976441955566406, metrics={'train_runtime': 1090.3418, 'train_samples_per_second': 2.009, 'train_steps_per_second': 0.252, 'total_flos': 72035058481920.0, 'train_loss': 0.2976441955566406, 'epoch': 5.0})

In [34]:
predictions = trainer.predict(test_dataset)

logits = predictions.predictions

# сигмоида
probs = 1 / (1 + np.exp(-logits))

print("Min prob:", probs.min())
print("Max prob:", probs.max())
print("Mean prob:", probs.mean())

threshold = 0.2 #модель не уверена в себе и на 0.5 выдает 0.0
y_pred_bert = (probs > threshold).astype(int)

if y_pred_bert.sum() == 0:
    print("!!! Все предсказания нули -> argmax fallback") #костыль если модель вообще ничего не предсказада
    y_pred_bert = np.zeros_like(probs)
    y_pred_bert[np.arange(len(probs)), probs.argmax(axis=1)] = 1

f1_micro_bert = f1_score(y_test, y_pred_bert, average="micro")
f1_macro_bert = f1_score(y_test, y_pred_bert, average="macro")

print("BERT F1 micro:", round(f1_micro_bert, 3))
print("BERT F1 macro:", round(f1_macro_bert, 3))

Min prob: 0.011770874
Max prob: 0.73780733
Mean prob: 0.13045444
BERT F1 micro: 0.422
BERT F1 macro: 0.181


### Маленькие выводы:
- Модель **ХУЖЕ** TF-IDF
- редкие классы почти не угадываются, игнорирует их

Проблемы могут быть в:
- маленьком количестве данных
- размере самих ФЕ (короткие, мало контекста)
- Multi-label + дисбаланс

Пока напрашивается вывод, что для данной задачи умная модель ≠ лучшая модель, но проверим еще кое-что...

## Модель XLM-RoBERTa с использованием дефиниций (подсказок)

Здесь к исходному тексту добавляются определения категорий.

Это позволяет:
- явно вводить семантическую информацию
- помогать модели при редких классах
- улучшать обобщающую способность

Формат входа:
[ТЕКСТ] + [SEP] + [ДЕФИНИЦИИ]

In [35]:
df_def = df.dropna(subset=["text", "definition", "labels", "lang"]).copy()

df_def["text"] = df_def["text"].astype(str)
df_def["definition"] = df_def["definition"].astype(str)
df_def["labels"] = df_def["labels"].astype(str)
df_def["lang"] = df_def["lang"].astype(str)

# объединяем
df_def["input_text"] = (
    df_def["text"] + " [SEP] " + df_def["definition"]
)

# метки
df_def["labels_list"] = df_def["labels"].apply(
    lambda x: [l.strip() for l in x.split(";") if l.strip()]
)

# фильтры
df_def = df_def[df_def["input_text"].str.len() > 0]
df_def = df_def[df_def["labels_list"].map(len) > 0]

# русский
df_def_ru = df_def[df_def["lang"] == "ru"].copy()

In [36]:
train_df_def, test_df_def = train_test_split(
    df_def_ru,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

mlb_def = MultiLabelBinarizer()
y_train_def = mlb_def.fit_transform(train_df_def["labels_list"])
y_test_def = mlb_def.transform(test_df_def["labels_list"])

In [37]:
train_dataset_def = TextDataset(train_df_def["input_text"], y_train_def)
test_dataset_def = TextDataset(test_df_def["input_text"], y_test_def)

In [38]:
model_bert_def = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(mlb_def.classes_),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
trainer_def = Trainer(
    model=model_bert_def,
    args=training_args,
    train_dataset=train_dataset_def
)

trainer_def.train()

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
50,0.556030
100,0.401045
150,0.366001
200,0.353814


TrainOutput(global_step=200, training_loss=0.41922242164611817, metrics={'train_runtime': 1093.1227, 'train_samples_per_second': 1.459, 'train_steps_per_second': 0.183, 'total_flos': 73777344785100.0, 'train_loss': 0.41922242164611817, 'epoch': 5.0})

In [40]:
predictions_def = trainer_def.predict(test_dataset_def)

logits_def = predictions_def.predictions
probs_def = 1 / (1 + np.exp(-logits_def))

threshold = 0.2

y_pred_def = (probs_def > threshold).astype(int)

if y_pred_def.sum() == 0:
    y_pred_def = np.zeros_like(probs_def)
    y_pred_def[np.arange(len(probs_def)), probs_def.argmax(axis=1)] = 1

f1_micro_bert_def = f1_score(y_test_def, y_pred_def, average="micro")
f1_macro_bert_def = f1_score(y_test_def, y_pred_def, average="macro")

print("BERT + definitions F1 micro:", round(f1_micro_bert_def, 3))
print("BERT + definitions F1 macro:", round(f1_macro_bert_def, 3))

BERT + definitions F1 micro: 0.329
BERT + definitions F1 macro: 0.126


C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Невероятно, но факт
Результат несколько контринтуитивный ("TF-IDF ведь выиграл от дефиниций! Что с тобой не так, BERT?!")
```
final train_loss ≈ 0.42
```
Это **хуже**, чем было без дефиниций (~0.29) -> модель обучается хуже, хотя информации больше.
Возможные причины:
- модель запуталась
```фразеологизм (сложный, образный) vs. дефиниция (простая, буквальная)```
- формат input плохой (не обучалась на "текст + его объяснение)
- мало данных
- дефиниции могут шуметь ("доживать последние дни жизни...")

#### BERT слишком много думает и сомневается в себе 💔 

# Итоги

In [47]:
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_macro_def = f1_score(y_test_def, y_pred_def, average="macro")

results = pd.DataFrame([
    ["TF-IDF", f1_micro, f1_macro],
    ["TF-IDF + дефиниции", f1_micro_def, f1_macro_def],
    ["BERT", f1_micro_bert, f1_macro_bert],
    ["BERT + дефиниции", f1_micro_bert_def, f1_macro_bert_def],
], columns=["Модель", "F1 micro", "F1 macro"])

results = results.round(3)

results

C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Модель,F1 micro,F1 macro
0,TF-IDF,0.473,0.354
1,TF-IDF + дефиниции,0.577,0.126
2,BERT,0.422,0.181
3,BERT + дефиниции,0.329,0.126


## Сравнение моделей и анализ результатов

| Модель              | F1 micro | F1 macro |
|---------------------|----------|----------|
| TF-IDF              | 0.473    | 0.354    |
| TF-IDF + дефиниции  | 0.577    | 0.126    |
| BERT                | 0.422    | 0.181    |
| BERT + дефиниции    | 0.329    | 0.126    |

### Что видно

- **TF-IDF с дефинициями — лучший результат по F1 micro (0.577)** - простая модель сильно выигрывает от явных слов-подсказок

- **BERT без дефиниций лучше, чем с ними** - добавление определений не помогло, а ухудшило качество

- **Macro F1 везде ниже micro** - модель плохо справляется с редкими классами (дисбаланс данных)

---

### Почему так получилось

- TF-IDF опирается на слова -> дефиниции добавляют полезные маркеры  
- BERT пытается понимать смысл, но:
  - формат "текст + определение" для него непривычен  
  - данных мало для такой сложной задачи  
  - дефиниции могут добавлять шум

---

### Главный вывод

- Простые модели выигрывают от явных подсказок  
- Сложные модели требуют больше данных и аккуратной постановки задачи  
- В этой задаче **TF-IDF оказался сильным baseline**, а BERT не раскрылся

---

### Итог

Добавление дефиниций:
- помогает TF-IDF  
- не помогает (и даже мешает) BERT  

# Бонус -- перенос моделей, обученных на русском языке, на английский и сербский.

## Без дефиниций

In [48]:
def evaluate_on_language(df_lang, name, mlb, model, threshold=0.2):
    df_lang = df_lang.copy()

    # фильтруем только известные метки
    df_lang["labels_list"] = df_lang["labels_list"].apply(
        lambda x: [l for l in x if l in mlb.classes_] if x else []
    )

    df_lang = df_lang[df_lang["labels_list"].map(len) > 0]

    if len(df_lang) == 0:
        print(f"{name}: нет подходящих данных")
        return None, None

    # бинализация
    y_true = mlb.transform(df_lang["labels_list"])

    # тексты
    df_lang["input_text"] = df_lang["text"]

    dataset = TextDataset(df_lang["input_text"], y_true)

    predictions = trainer.predict(dataset)
    logits = predictions.predictions

    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs > threshold).astype(int)

    if y_pred.sum() == 0:
        y_pred = np.zeros_like(probs)
        y_pred[np.arange(len(probs)), probs.argmax(axis=1)] = 1

    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")

    print(f"{name} F1 micro:", round(f1_micro, 3))
    print(f"{name} F1 macro:", round(f1_macro, 3))

    return f1_micro, f1_macro


# дележка по языкам
df_en_test = df[df["lang"] == "en"]
df_sr_test = df[df["lang"] == "sr"]

# запуск
f1_en_micro, f1_en_macro = evaluate_on_language(df_en_test, "EN", mlb, model_bert)
f1_sr_micro, f1_sr_macro = evaluate_on_language(df_sr_test, "SR", mlb, model_bert)

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


EN F1 micro: 0.297
EN F1 macro: 0.129


C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


SR F1 micro: 0.367
SR F1 macro: 0.158


C:\Users\User\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## С дефинициями

In [73]:
def evaluate_on_language_def(df_lang, name, mlb, trainer, threshold=0.2):
    df_lang = df_lang.copy()

    df_lang["labels_list"] = df_lang["labels_list"].apply(
        lambda x: [l for l in x if l in mlb.classes_] if x else []
    )

    df_lang = df_lang[df_lang["labels_list"].map(len) > 0]

    if len(df_lang) == 0:
        print(f"{name} + DEF: нет подходящих данных")
        return None, None

    df_lang["definition"] = df_lang["definition"].fillna("")

    df_lang["input_text"] = df_lang["text"] + " [SEP] " + df_lang["definition"]

    y_true = mlb.transform(df_lang["labels_list"])

    dataset = TextDataset(df_lang["input_text"], y_true)

    predictions = trainer.predict(dataset)
    logits = predictions.predictions

    probs = 1 / (1 + np.exp(-logits))
    y_pred = (probs > threshold).astype(int)

    if y_pred.sum() == 0:
        y_pred = np.zeros_like(probs)
        y_pred[np.arange(len(probs)), probs.argmax(axis=1)] = 1

    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"{name} + DEF F1 micro:", round(f1_micro, 3))
    print(f"{name} + DEF F1 macro:", round(f1_macro, 3))

    return f1_micro, f1_macro

In [74]:
df_en_test = df[df["lang"] == "en"]
df_sr_test = df[df["lang"] == "sr"]

f1_en_micro_def, f1_en_macro_def = evaluate_on_language_def(
    df_en_test, "EN", mlb, trainer
)

f1_sr_micro_def, f1_sr_macro_def = evaluate_on_language_def(
    df_sr_test, "SR", mlb, trainer
)

C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


EN + DEF F1 micro: 0.279
EN + DEF F1 macro: 0.099


C:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


SR + DEF F1 micro: 0.288
SR + DEF F1 macro: 0.122


## Интерпретация результатов

### Модель без дефиниций

| Язык | F1 micro | F1 macro |
|------|---------|----------|
| EN   | 0.297   | 0.129    |
| SR   | 0.367   | 0.158    |

### Модель с дефинициями

| Язык | F1 micro | F1 macro |
|------|---------|----------|
| EN   | 0.279   | 0.099    |
| SR   | 0.288   | 0.122    |

---

### Сравнение результатов

**Английский (EN):**
- F1 micro: **0.297 → 0.279** (↓)
- F1 macro: **0.129 → 0.099** (↓)

**Сербский (SR):**
- F1 micro: **0.367 → 0.288** (↓ сильно)
- F1 macro: **0.158 → 0.122** (↓)

Сербский переносится лучше, чем английский (логично)
Во всех случаях добавление дефиниций **ухудшило качество модели**.

---

### Интерпретация

И в данном случае добавление дефиниций только испортило качество, что, впрочем, было ожидаемо. Изначально (до первого эксперимента на русском языке) наивно ожидалось, что дополнительные определения помогут модели лучше понимать термины и, соответственно, точнее классифицировать тексты. Но на практике этого не произошло.

Скорее всего, дефиниции:
- либо добавляют лишний шум,
- либо перегружают входные данные,
- либо просто не используются моделью так, как надо.


Но главное наблюдение - во всех случаях (и EN, и SR) качество падает после добавления дефиниций, причём для сербского падение даже более заметное.

### Вывод

Добавление дефиниций в текущем виде не улучшает модель и использовать их в таком формате не имеет смысла.

# 🎲 Фан-Бонус: генерация фразеологизмов

Чтобы немного отвлечься от  метрик, я попробовала сгенерировать новые фразеологизмы на основе своего датасета.

Сначала был простой символьный Markov-генератор, но он часто выдавал мусор.
В финальной версии (v3) я добавила:

- шаблоны (типичные структуры фразеологизмов)
- морфологию (согласование слов)
- фильтры (убираем странные и редкие слова)

В итоге получился генератор, который иногда выдает вполне "похожее на язык"

In [75]:
import re
import random
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

In [76]:
path = r"C:\Users\User\Питон Вышка все файлы\словари\txt\РЯ\смрт\done\Русская_смерть_ручная.csv"
df = pd.read_csv(path)

phrases = []

for text in df["Фразеологизм"].dropna():
    parts = str(text).split("/")

    for p in parts:
        p = p.strip().lower()

        p = re.sub(r"[^а-яё\s-]", "", p)
        p = re.sub(r"\s+", " ", p).strip()

        if len(p) > 3:
            phrases.append(p)

print(f"Всего фразеологизмов: {len(phrases)}")

tokenized = [p.split() for p in phrases]

word_freq = Counter(w for p in tokenized for w in p)


def is_rare(word):
    return word_freq[word] < 2

trash_words = {
    "и", "да", "а", "что", "как", "кто",
    "кого", "кому", "чего", "нибудь"
}

def is_trash(word):
    return word in trash_words or len(word) <= 2

VALID_POS = {
    "NOUN", "VERB", "INFN", "ADJF", "ADJS",
    "ADVB", "NPRO", "PRED", "PRTF", "PRTS",
    "GRND", "NUMR"
}

def safe_get_pos(word):
    try:
        p = morph.parse(word)[0].tag.POS
        if p is None:
            return None

        p = str(p)
        if p not in VALID_POS:
            return None

        return p
    except:
        return None

pos_to_words = {}

for word in word_freq:
    pos = safe_get_pos(word)
    if pos:
        pos_to_words.setdefault(pos, []).append(word)

def inflect_word(word, target_case):
    parsed = morph.parse(word)[0]

    if parsed.tag.POS != "NOUN":
        return word

    try:
        form = parsed.inflect({target_case})
        if form:
            return form.word
    except:
        pass

    return word

templates = [

    [
        ("VERB", None),
        ("PREP", "под"),
        ("NOUN", "accs")
    ],

    [
        ("VERB", None),
        ("NOUN", "ablt")
    ],

    [
        ("NOUN", None),
        ("VERB", None)
    ],

    [
        ("VERB", None),
        ("ADJF", None),
        ("NOUN", "ablt")
    ],

    [
        ("VERB", None),
        ("PREP", "на"),
        ("ADJF", None),
        ("NOUN", "accs")
    ]
]

def generate_phrase_v3():

    template = random.choice(templates)
    words = []

    for pos, extra in template:

        if pos == "PREP":
            words.append(extra)
            continue

        candidates = [
            w for w in pos_to_words.get(pos, [])
            if not is_rare(w) and not is_trash(w)
        ]

        if not candidates:
            return None

        word = random.choice(candidates)

        if pos == "NOUN" and extra:
            word = inflect_word(word, extra)

        words.append(word)

    return words

def has_repeats(words):
    return len(words) != len(set(words))


def bad_semantics(words):
    lemmas = [morph.parse(w)[0].normal_form for w in words]

    if "смерть" in lemmas and "умереть" in lemmas:
        return True

    return False


def is_good_v3(words):

    if words is None:
        return False

    if has_repeats(words):
        return False

    if any(is_trash(w) for w in words):
        return False

    if bad_semantics(words):
        return False

    return True

generated_v3 = []
attempts = 0

while len(generated_v3) < 100 and attempts < 10000:
    words = generate_phrase_v3()

    if is_good_v3(words):
        phrase = " ".join(words)
        generated_v3.append(phrase)

    attempts += 1

output_path_v3 = "de_generated_phrases_v3.txt"

with open(output_path_v3, "w", encoding="utf-8") as f:
    for line in generated_v3:
        f.write(line + "\n")

print(f"\nСгенерировано: {len(generated_v3)} фраз")
print("Файл сохранён:", output_path_v3)


print("\n🎲 Первые 10 сгенерированных фраз:\n")

for i, phrase in enumerate(generated_v3[:10], 1):
    print(f"{i}. {phrase}")

# бонус: случайная "лучшая"
print("\n🔥 Случайный фразеологизм дня:")
print(random.choice(generated_v3))

Всего фразеологизмов: 674

Сгенерировано: 100 фраз
Файл сохранён: de_generated_phrases_v3.txt

🎲 Первые 10 сгенерированных фраз:

1. умрет под кровь
2. живет русскому веком
3. живет глазами
4. умел под живот
5. что-л бойся
6. стоит свой душой
7. стоит под дурака
8. пришла под кости
9. родится детьми
10. кажет душой

🔥 Случайный фразеологизм дня:
станешь находкой
